In [17]:
from pyiceberg.catalog import load_catalog
import pyarrow.fs as fs


catalog = load_catalog(
    "default",
    **{
        "type": "hive",
        "uri": "thrift://hive-metastore:9083",  # Lebih rapi pakai hostname
        "s3.endpoint": "http://minio:9000",      # Lebih rapi pakai hostname
        "s3.access-key-id": "minioadmin",
        "s3.secret-access-key": "minioadmin",
        "s3.use-ssl": "false",
    }
)

In [18]:
schema_name = "bronze"
table_name = "yellow_taxi"
bucket_name = "lakehouse"
table_id = f"{schema_name}.{table_name}"  # Menjadi 'bronze.yellow_taxi'
table_path = f"{bucket_name}/{schema_name}/{table_name}"

# 2. Inisialisasi S3 FileSystem (PyArrow)
s3_fs = fs.S3FileSystem(
    endpoint_override="http://minio:9000",  # Ganti jika beda port
    access_key="minioadmin",
    secret_key="minioadmin",
    scheme="http",
    region="us-east-1",
)

try:
    catalog.drop_table(table_id)
    print(f"🗑️ Tabel {table_id} berhasil dihapus dari katalog dan storage.")

    file_info = s3_fs.get_file_info(table_path)

    if file_info.type != fs.FileType.NotFound:
        s3_fs.delete_dir(table_path)
        print(f"🧹 Folder '{table_path}' di MinIO berhasil dibersihkan secara total.")
    else:
        print(f"ℹ️ Folder '{table_path}' sudah tidak ada di storage.")
except Exception as e:
    print(f"❌ Gagal menghapus tabel: {e}")

🗑️ Tabel bronze.yellow_taxi berhasil dihapus dari katalog dan storage.
🧹 Folder 'lakehouse/bronze/yellow_taxi' di MinIO berhasil dibersihkan secara total.
